# Notebook 5: Evaluation & SHAP Explainability

This notebook answers two questions:
1. **How do we know if the model is actually good?** — metrics beyond accuracy
2. **Why did the model make a specific prediction?** — SHAP explainability

---

## Why Evaluation Matters Beyond Accuracy

Accuracy tells you the overall % correct. But with 73 topics of different sizes, accuracy is a blunt instrument. Consider:

- A model that nails all big topics but fails on small ones → high accuracy, low F1
- A model that's systematically confused between two similar topics → accuracy looks fine, but the confusion matrix reveals the problem

**Good evaluation = accuracy + F1 + confusion matrix + explainability**

In [ ]:
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, precision_score, recall_score
)

sns.set_theme(style='whitegrid', palette='muted')

# Load model and reconstruct test set
model      = pickle.load(open('../data/processed/models/logistic_regression.pkl', 'rb'))
le         = pickle.load(open('../data/processed/models/label_encoder.pkl', 'rb'))
vectorizer = pickle.load(open('../data/processed/tfidf_vectorizer.pkl', 'rb'))

tfidf_full = sp.load_npz('../data/processed/tfidf_matrix.npz')
df_clean   = pd.read_parquet('../data/processed/articles_clean.parquet', columns=['id'])
df_clean['row_idx'] = np.arange(len(df_clean))
df_labeled = pd.read_parquet('../data/processed/articles_with_topics.parquet')
df_merged  = df_labeled.merge(df_clean, on='id', how='inner')

X = tfidf_full[df_merged['row_idx'].values, :]
y = le.transform(df_merged['topic_id'].values)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df_merged, test_size=0.2, random_state=42, stratify=y
)
y_pred = model.predict(X_test)
feature_names = vectorizer.get_feature_names_out()

print(f'Test set: {X_test.shape[0]:,} articles, {len(le.classes_)} topics')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 macro: {f1_score(y_test, y_pred, average="macro"):.4f}')

---

## Precision, Recall, and F1 — What Each Means

These three metrics come up in every ML interview. Let's define them with a concrete example from our model.

In [ ]:
# Pick one topic and walk through the 2x2 confusion table
# Let's use topic 0 (police/crime — the largest class)
topic_idx = 0
topic_label = le.classes_[topic_idx]

# Binary view: is this topic or not?
y_true_binary = (y_test == topic_idx).astype(int)
y_pred_binary = (y_pred == topic_idx).astype(int)

TP = ((y_true_binary == 1) & (y_pred_binary == 1)).sum()  # correctly predicted as this topic
FP = ((y_true_binary == 0) & (y_pred_binary == 1)).sum()  # predicted this topic, actually different
FN = ((y_true_binary == 1) & (y_pred_binary == 0)).sum()  # actually this topic, predicted different
TN = ((y_true_binary == 0) & (y_pred_binary == 0)).sum()  # correctly predicted as NOT this topic

precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f'Topic {topic_label} — Binary Confusion Table')
print(f'                    Predicted YES   Predicted NO')
print(f'  Actually YES:       {TP:>6} (TP)    {FN:>6} (FN)  ← missed these')
print(f'  Actually NO:        {FP:>6} (FP)    {TN:>6} (TN)')
print()
print(f'Precision = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision:.3f}')
print(f'  → Of all articles I said were topic {topic_label}, {precision:.1%} actually were')
print()
print(f'Recall    = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.3f}')
print(f'  → Of all actual topic {topic_label} articles, I caught {recall:.1%} of them')
print()
print(f'F1        = {f1:.3f}  (harmonic mean — punishes if either P or R is low)')

In [ ]:
# Show precision vs recall tradeoff across all topics
report_dict = classification_report(
    y_test, y_pred,
    target_names=[str(c) for c in le.classes_],
    output_dict=True
)
# Build per-class dataframe
per_class = []
for label in [str(c) for c in le.classes_]:
    if label in report_dict:
        per_class.append({
            'topic': label,
            'precision': report_dict[label]['precision'],
            'recall':    report_dict[label]['recall'],
            'f1':        report_dict[label]['f1-score'],
            'support':   report_dict[label]['support'],
        })
per_class_df = pd.DataFrame(per_class).sort_values('f1')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(per_class_df))
ax.plot(x, per_class_df['precision'], 'o-', label='Precision', color='steelblue', markersize=4)
ax.plot(x, per_class_df['recall'],    's-', label='Recall',    color='coral',     markersize=4)
ax.plot(x, per_class_df['f1'],        '^-', label='F1',        color='green',     markersize=4)
ax.set_xticks(x)
ax.set_xticklabels(per_class_df['topic'], rotation=60, ha='right', fontsize=7)
ax.set_title('Precision / Recall / F1 per Topic (sorted by F1)', fontsize=13)
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

print('Hardest topics (lowest F1):')
print(per_class_df[['topic','precision','recall','f1','support']].head(5).to_string(index=False))

---

## The Confusion Matrix

The confusion matrix shows **which topics get confused with each other**. A normalized matrix (rows sum to 1.0) lets you see: "of all articles that were actually topic X, what fraction did the model predict as each topic?"

In [ ]:
# Full normalized confusion matrix
cm = confusion_matrix(y_test, y_pred, normalize='true')
labels = [str(c) for c in le.classes_]

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=labels, yticklabels=labels,
    ax=ax, vmin=0, vmax=1,
    annot_kws={'size': 5},
)
ax.set_title('Normalized Confusion Matrix (row = true topic, value = fraction predicted as each topic)', fontsize=11)
ax.set_ylabel('True Topic')
ax.set_xlabel('Predicted Topic')
plt.xticks(rotation=45, ha='right', fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# Find the worst confusions — pairs of topics most often mixed up
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
np.fill_diagonal(cm_df.values, 0)  # zero out the diagonal (correct predictions)

# Stack into pairs and sort
confusion_pairs = (
    cm_df.stack()
    .reset_index()
    .rename(columns={'level_0': 'true_topic', 'level_1': 'predicted_as', 0: 'fraction'})
    .sort_values('fraction', ascending=False)
    .head(10)
)
print('Top 10 most common confusions (true → predicted):')
print(confusion_pairs.to_string(index=False))
print()
print('These pairs share vocabulary — e.g. two political topics both mention "trump" and "president".')

> **Interview Talking Point:**  
> *"The confusion matrix revealed that the model most often confuses topics with overlapping vocabulary — for example, multiple political topics that all mention 'trump', 'president', and 'senate'. This kind of error analysis tells you where to invest next: either merge similar topics, add features that distinguish them, or accept the confusion as inherent topic overlap."*

---

## SHAP — Why Did the Model Predict This Topic?

### What is SHAP?

SHAP stands for **SHapley Additive exPlanations**. It answers: *"how much did each feature contribute to this specific prediction?"*

The name comes from game theory. Shapley values were invented in the 1950s to fairly divide the "credit" among players in a cooperative game. Applied to ML:

- The **game** = making a prediction
- The **players** = features (words in our case)
- The **payout** = how much the prediction score for the predicted class changed vs the average

A positive SHAP value for word "federal" on topic 1 means: *"the presence of 'federal' in this article pushed the model toward topic 1."*

In [ ]:
# ── SHAP for linear models — the key insight ────────────────────────────────
# For Logistic Regression, SHAP values have a simple formula:
#
#   SHAP(feature j, class k) = coef[k, j] × (x_j - mean(x_j))
#
# Where:
#   coef[k, j]    = the weight the model learned for word j when predicting topic k
#   x_j           = the TF-IDF score of word j in THIS article
#   mean(x_j)     = the average TF-IDF score of word j across ALL articles (background)
#
# For TF-IDF, most values are 0 (sparse), so mean(x_j) ≈ 0
# This means: SHAP(j, k) ≈ coef[k, j] × x_j
# → Words with high TF-IDF score in this article AND high coefficient for this topic
#   have the highest SHAP value.

print('SHAP value = coefficient × (article value - background mean)')
print()
print('Example:')
print('  Word "federal":')
print('    coef for topic 1 (finance) = +2.3')
print('    TF-IDF in this article     = 0.45')
print('    Background mean            ≈ 0.003')
print('    SHAP value                 = 2.3 × (0.45 - 0.003) ≈ +1.03')
print()
print('  Interpretation: "federal" pushed the prediction toward topic 1 by +1.03 units')

In [ ]:
# ── Global importance: use model.coef_ directly ──────────────────────────────
# For a linear model, the coefficient for word j in class k IS the global
# feature importance. It tells us: across all articles, a one-unit increase
# in the TF-IDF score of word j pushes the prediction toward class k by coef[k,j].

# Let's look at the top words for a few topics
topics_to_show = [0, 1, 2, 3]  # first 4 topics
topic_info = pd.read_csv('../data/processed/topic_info.csv')
topic_names = dict(zip(topic_info['Topic'], topic_info['Name']))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for plot_idx, class_idx in enumerate(topics_to_show):
    coefs    = model.coef_[class_idx]
    top_idx  = np.argsort(coefs)[::-1][:12]
    words    = feature_names[top_idx]
    scores   = coefs[top_idx]
    topic_id = le.classes_[class_idx]

    ax = axes[plot_idx]
    ax.barh(words[::-1], scores[::-1], color=sns.color_palette('muted')[plot_idx])
    name = topic_names.get(topic_id, str(topic_id))
    ax.set_title(f'Topic {topic_id}\n{name[:35]}', fontsize=9)
    ax.set_xlabel('Coefficient (≈ SHAP)', fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('Top Words per Topic — model.coef_ (equivalent to SHAP for linear models)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP for a SINGLE article — the real explainability use case ─────────────
# Global coefficients tell us what the model learned in general.
# SHAP for a single article tells us: for THIS specific article,
# which words actually appeared and drove this specific prediction?

import shap

# Pick a correctly-classified article
correct_mask = (y_pred == y_test)
article_idx  = np.where(correct_mask)[0][0]

true_class = le.classes_[y_test[article_idx]]
pred_class = le.classes_[y_pred[article_idx]]
pred_class_idx = y_pred[article_idx]

# Get non-zero features for this article (typically 100-500 out of 50k)
article_vec  = X_test[article_idx]
nonzero_idx  = article_vec.nonzero()[1]
print(f'Article has {len(nonzero_idx)} non-zero TF-IDF features out of {X_test.shape[1]:,} total')
print(f'True topic: {true_class}  |  Predicted: {pred_class}')
print()

# Build background (100 training articles, only non-zero feature columns)
np.random.seed(42)
bg_idx      = np.random.choice(X_train.shape[0], size=100, replace=False)
X_bg_dense  = X_train[bg_idx][:, nonzero_idx].toarray()
X_art_dense = article_vec[:, nonzero_idx].toarray()

# Slice model coefficients to match the feature subset
from sklearn.linear_model import LogisticRegression as LR
sub_model             = LR()
sub_model.coef_       = model.coef_[:, nonzero_idx]
sub_model.intercept_  = model.intercept_
sub_model.classes_    = model.classes_

explainer   = shap.LinearExplainer(sub_model, X_bg_dense)
shap_values = explainer.shap_values(X_art_dense)  # list: one array per class

sv          = shap_values[pred_class_idx][0]       # SHAP values for predicted class
sub_features = feature_names[nonzero_idx]

# Top pushing-toward features
top_pos = np.argsort(sv)[::-1][:10]
top_neg = np.argsort(sv)[:10]
show    = np.concatenate([top_neg[::-1], top_pos[::-1]])

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['coral' if sv[i] < 0 else 'steelblue' for i in show]
ax.barh(sub_features[show], sv[show], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(
    f'SHAP Single-Article Explanation\nTrue: topic {true_class}  |  Predicted: topic {pred_class}',
    fontsize=11
)
ax.set_xlabel('SHAP value  (blue = pushes toward predicted topic, red = pushes away)')
plt.tight_layout()
plt.show()

print('Blue words: pushed the model TOWARD the predicted topic')
print('Red words:  pushed the model AWAY from the predicted topic')

### Why does SHAP matter for an interview?

Imagine showing this pipeline to a non-technical stakeholder — a news editor or product manager. They ask: *"Why did your model label my article as 'US Politics'?"*

Without SHAP: *"Because the model predicted that class."* — useless.

With SHAP: *"The words 'senate', 'filibuster', and 'legislation' strongly pushed it toward US Politics. The word 'economy' tried to pull it toward Finance, but the political terms dominated."*

> **Interview Talking Point:**  
> *"I used SHAP to surface which words drove each topic prediction. For our linear model, SHAP values are the product of the learned coefficient and the article's TF-IDF score for that word — so they're both mathematically grounded and easy to communicate. This is important for debugging: if a topic is being systematically misclassified, SHAP shows you exactly what vocabulary the model is keying on."*

---

## Why model.coef_ ≈ SHAP for This Model

In [ ]:
# Demonstrate the equivalence on a single article
# SHAP(j, k) = coef[k,j] × (x_j - mean_bg_j)

# Manual SHAP calculation for the first 5 non-zero features
mean_bg = X_bg_dense.mean(axis=0)  # background mean for each non-zero feature

print('Manual SHAP vs shap.LinearExplainer output (should match):')
print()
print(f'{"Word":<25} {"TF-IDF":>8} {"Coef":>8} {"Manual SHAP":>12} {"Library SHAP":>13}')
print('-' * 70)

for i in np.argsort(sv)[::-1][:5]:
    word       = sub_features[i]
    x_val      = X_art_dense[0, i]
    coef_val   = sub_model.coef_[pred_class_idx, i]
    manual     = coef_val * (x_val - mean_bg[i])
    library    = sv[i]
    print(f'{word:<25} {x_val:>8.4f} {coef_val:>8.4f} {manual:>12.4f} {library:>13.4f}')

print()
print('They match! For linear models, SHAP is just: coef × (value - background_mean)')

---

## Summary

| Concept | What it tells you |
|---------|-------------------|
| Accuracy | Overall % correct — misleading with class imbalance |
| Precision | Of predictions for topic X, how many were right? |
| Recall | Of actual topic X articles, how many did we catch? |
| F1 macro | Average F1 across all topics — primary metric |
| Confusion matrix | Which topics get mixed up and why |
| model.coef_ | Global word importance per topic (= SHAP for linear models) |
| SHAP single-article | Which words drove THIS specific prediction |

**Key insight:** For a linear model like Logistic Regression, model coefficients and SHAP values are mathematically equivalent when features are zero-centered. This means explainability is essentially free for LR — no extra computation needed for global importance.